<a href="https://colab.research.google.com/github/BrundaSreedhar/awesome-ai-experiments/blob/main/Experiments_Using_FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install faiss-cpu numpy sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 47.0 MB/s eta 0:00:00


In [4]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

In [5]:
documents = [
    "Kubernetes Pods are the smallest deployable units.",
    "A Kubernetes Service provides stable network access to Pods.",
    "A Deployment manages ReplicaSets and maintains the desired number of Pods.",
    "FAISS is a library for efficient similarity search over vectors.",
]

# 1. Create embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(documents)

# 2. Convert to FAISS-compatible float32
embeddings = np.array(embeddings).astype("float32")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# 3. Create an index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

# 4. Add vectors
index.add(embeddings)

In [8]:
# 5. Embed a query
query = "What is a pod?"
query_embedding = model.encode([query]).astype("float32")

# 6. Search for nearest vectors
k = 2
distances, indices = index.search(query_embedding, k)

# 7. Retrieve original documents
for i in indices[0]:
    print(documents[i])

A Deployment manages ReplicaSets and maintains the desired number of Pods.
A Kubernetes Service provides stable network access to Pods.


FAISS lets you do semantic search over documents

01. Load paper
       ↓
02. Parse text + metadata
       ↓
03. Chunk paper
       ↓
04. Generate embeddings
       ↓
05. Build FAISS index
       ↓
06. Ask test questions
       ↓
07. Retrieve top-k chunks
       ↓
08. Inspect retrieval quality
       ↓
09. Generate grounded answer
       ↓
10. Evaluate retrieval

In [9]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 2.4 MB/s eta 0:00:00


In [10]:
from pypdf import PdfReader

In [13]:
#Load the PDF
pdf_path = "attention.pdf" #i uploaded this manually, we could also scrape one off

reader = PdfReader(pdf_path)

pages = []

for page_num, page in enumerate(reader.pages):
    text = page.extract_text()

    pages.append({
        "page": page_num + 1,
        "text": text
    })

print(f"Loaded {len(pages)} pages")

Loaded 15 pages


In [14]:
print(pages[5]["text"][:2000])

Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations
for different layer types.n is the sequence length,d is the representation dimension,k is the kernel
size of convolutions andr the size of the neighborhood in restricted self-attention.
Layer Type Complexity per Layer Sequential Maximum Path Length
Operations
Self-Attention O(n2·d) O(1) O(1)
Recurrent O(n·d2) O(n) O(n)
Convolutional O(k·n·d2) O(1) O(logk(n))
Self-Attention (restricted) O(r·n·d) O(1) O(n/r)
3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of the
order of the sequence, we must inject some information about the relative or absolute position of the
tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the
bottoms of the encoder and decoder stacks. The positional encodings have the same dimensiondmodel
as the embeddings, so that the two can be summed. There are many choice

Create a very simple chunker

In [15]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks

In [16]:
chunks = []

for page in pages:
    page_chunks = chunk_text(page["text"])

    for i, chunk in enumerate(page_chunks):
        chunks.append({
            "text": chunk,
            "page": page["page"],
            "chunk_id": f"page_{page['page']}_chunk_{i}"
        })

print(len(chunks))

59


Generate Embeddings

In [17]:
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    texts,
    normalize_embeddings=True
)

embeddings = np.asarray(embeddings).astype("float32")

print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(59, 384)


Build FAISS

Because we're normalizing the vectors, we can use inner product as cosine similarity:

In [18]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print(index.ntotal)

59


In [19]:
questions = [
    "What problem are the authors trying to solve?",
    "Why did the authors choose this architecture?",
    "What dataset did they use?",
    "What are the main limitations of the approach?",
    "How does their approach compare to previous work?",
    "What does the loss function represent?",
    "What were the main experimental results?"
]

In [20]:
def retrieve(question, k=5):
    query_embedding = model.encode(
        [question],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(query_embedding).astype("float32")

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        result = chunks[idx].copy()
        result["score"] = float(score)

        results.append(result)

    return results

In [25]:
results = retrieve(
    "What is positional encoding?",
    k=5
)

In [26]:
for result in results:
    print("=" * 80)
    print(
        f"Page: {result['page']} | "
        f"Score: {result['score']:.3f}"
    )
    print(result["text"][:1000])

Page: 6 | Score: 0.578
to the input embeddings at the
bottoms of the encoder and decoder stacks. The positional encodings have the same dimensiondmodel
as the embeddings, so that the two can be summed. There are many choices of positional encodings,
learned and fixed [9].
In this work, we use sine and cosine functions of different frequencies:
PE (pos,2i) =sin(pos/100002i/dmodel)
PE (pos,2i+1) =cos(pos/100002i/dmodel)
wherepos is the position andi is the dimension. That is, each dimension of the positional encoding
corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000· 2π. We
chose this function because we hypothesized it would allow the model to easily learn to attend by
relative positions, since for any fixed offsetk,PE pos+k can be represented as a linear function of
PE pos.
We also experimented with using learned positional embeddings [9] instead, and found that the two
versions produced nearly identical results (see Table 3 row (E)). We chose the